# Inlämningsuppgift 2: Utgående långvågig strålning (OLR)

## Introduktion

I Inlämningsuppgift 1 använde du enkla energibalansmodeller för att studera jordytans och atmosfärens temperatur vid strålningsbalans för olika parametervärden.
Här undersöker vi hur strålningsbalansen påverkas när koncentrationen av olika växthusgaser förändras i atmosfären.
Du kommer att simulera utgående långvågig strålning (OLR, outgoing longwave radiation) och analysera den momentana förändringen när du varierar växthusgaskoncentration.
I verkligheten skulle denna obalans resultera i temperaturförändringar som driver systemet mot ett nytt jämviktstillstånd.

Ett av målen med uppgiften är att uppskatta den *strålningsdrivning* (radiative forcing) som utsläpp av olika växthusgaser ger upphov till.
IPCC beräknar strålningsdrivning med avancerade klimat- och strålningstransportmodeller, se Figur 1 nedan.
I den här övningen använder vi en förenklad strålningstransportmodell.
Kan vi med denna enkla modell reproducera delar av denna figur?

<figure>
    <img src="./media/ipcc_5_ar6.png" width="800">
    <figcaption>
        <b>Figur 1: </b>
        Uppskattning av strålningsdrivning från olika klimatpåverkande faktorer för perioden 1750–2019.
        Siffrorna till höger anger bästa uppskattningar med osäkerhetsintervall (5–95%) inom hakparentes.
        Från IPCC AR6, 2021: <em>Climate Change 2021: The Physical Science Basis</em>.
    </figcaption>
</figure>

Uppgiften ger er också praktisk träning i att:

- Utföra numeriska beräkningar i Python
- Arbeta med flerdimensionella datastrukturer (matriser, eller mer generellt arrays)
- Visualisera resultat med tydliga figurer

## Avgränsningar

Simuleringarna är ytterst detaljerade när det gäller gasernas absorptionsspektra.
Men för att göra problemet hanterbart så bortser vi ifrån moln.
Vi antar även att marken agerar som en svartkropp och att atmosfären inte har någon horisontell variation (båda är goda approximationer).
Datan som du får täcker höjder upp till 20 km och representerar tropiska förhållande med förindustriella gasmängder.

## Teori

Modellen vi använder beräknar den **spektrala radiansen**,
$I$ (enhet: Wm<sup>-2</sup>Hz<sup>-1</sup>sr<sup>-1</sup>, där sr står för steradian),
för givna frekvenser $\nu$ och en given zenitvinkel $\theta$.
En zenitvinkel på 0° motsvarar kortaste vägen genom atmosfären,
medan en större zenitvinkel ger en längre väg,
se Figur 2 nedan.
I allmänhet beror $I$ även på azimutvinkel $\phi$,
men detta bortses från i modellen (se Avgränsningar).

<figure>
   <img src="./media/zenith-azimuth-schematic.svg" width=100%>
    <figcaption>
        <b>Figur 2: </b>(a) Illustration av utgående spektral radians för en viss vinkel.
        (b) Mer detaljerad bild av zenit- och azimutvinkel för utgående spektral radians.
    </figcaption>
</figure>

För att beräkna jordens **spektrala emittans**,
$E_s$ (enhet: Wm<sup>-2</sup>Hz<sup>-1</sup>),
behöver vi integrera den spektrala radiansen som strålar uppåt från atmosfären,
det vill säga över en halvsfär vid toppen av atmosfären:

$$
E_s(\nu) = \int_{\phi=0}^{2\pi} \int_{\theta=0}^{\pi / 2} I(\nu, \theta, \phi)\cos(\theta)\sin(\theta) \, d\theta d\phi
$$

För de förhållanden som beskrivs under Avgränsningar så beror inte $I$ på azimutvinkel $\phi$,
och uttrycket kan då förenklas som:

$$
E_s(\nu) = \int_{\phi=0}^{2\pi} d\phi \cdot \int_{\theta=0} ^{\pi / 2} I(\nu, \theta) \cos(\theta) \sin(\theta) \, d\theta
$$

Den första integralen kan lösas analytiskt och ger ett konstant värde, $C$.
Beräkna integralen analytiskt.
Vad blir $C$?

$$
C = \int_{\phi = 0}^{2\pi} d\phi = ?
\tag{1}
$$

Den spektrala emittansen ges därmed av:

$$
E_s(\nu) = C \cdot \int_{\theta=0}^{\pi/2} I(\nu, \theta)\cos(\theta)\sin(\theta) \, d\theta
\tag{2}
$$

Jordens totala OLR eller **emittans**,
$E$,
erhålls genom att integrera den spektrala emittansen över alla frekvenser:

$$
E = \int_{v=0}^\infty E_s(\nu) \, d\nu
\tag{3}
$$

En av dina uppgifter är att beräkna integralerna i Ekvation (2) och (3) med hjälp av numeriska metoder.

### Frekvens och vågtal
I figurer med strålning är det vanligt att frekvensen anges som “vågtal”, med enhet cm<sup>−1</sup>.
Detta vågtal är $1/\lambda$ där $\lambda$ är våglängden i cm.
Som exempel så visas Planckfunktionen som en funktion av vågtal i Figur 3.

<figure>
    <img src="./media/planck.png" width=600>
    <figcaption>
        <b>Figur 3: </b>Planck-kurvorna för två olika temperaturer som funktion av vågtal.
    </figcaption>
</figure>

## Kod och data

Funktioner för att beräkna spektral radians finns i modulen `olr.py` i mappen `Kod`.
Datan kommer i två varianter i mappen `Data`,
där filerna heter `olr_data.npz` och `olr_data large.npz`.
Båda dessa filer innehåller följande variabler:

| | | |
|-|-|-|
| **f**    | Frekvenser [Hz]               | En vektor (endimensionell array) med längd $m$ (antalet frekvenser) | 
| **wn**   | Vågtal [cm<sup>-1</sup>]      | En vektor med längd $m$ | 
| **z**    | Höjder [m]                    | En vektor med längd $k$ (antalet höjder) | 
| **p**    | Atmosfäriskt tryck [Pa]       | En vektor med längd $k$ |
| **t**    | Atmosfäriskt temperatur [K]   | En vektor med längd $k$ |
| **vmr**  | Volymandelar [-]              | En matris med dimensioner (gas, z). Inkluderade gaser, i ordning, är:<br>H<sub>2</sub>O, CO<sub>2</sub>, O<sub>3</sub>, CH<sub>4</sub> och N<sub>2</sub>O |
| **xsec**  |Absorptionstvärsnitt [m<sup>2</sup>] |En array med dimensioner (gas, f, z). Denna variabel kommer ni inte<br>använda direkt, den används av funktioner som beräknar radians |

Använd först och främst `olr_data.npz`, som innehåller data för 3500 frekvenser.
Om du är nyfiken och vill se resultaten i en högre upplösning kan du byta till `olr_data_large.npz` när du är klar med uppgiften.
Den här filen innehåller data för 35000 frekvenser.

## Kodutveckling

Nedan ges praktiska instruktioner för att hjälpa dig utveckla koden för uppgiften.

### Steg 1

Importera NumPy som `np`:

In [ ]:
# Importera numpy här

Ladda sedan in datafilen `Data/olr_data.npz` med NumPy-funktionen `load` till en variabel med namnet `data`.
Läs dokumentationen för `load` om du inte vet hur den fungerar, t.ex. med `help(np.load)`.

In [ ]:
# Ladda in data till en variabel med namnet data

Med följande syntax tilldelar vi frekvernsdatan till variablen `f`:

In [ ]:
f = data["f"]

Använda liknande syntax för att skapa en variabel för varje fysisk variabel i datan, se tabellen ovan.

In [ ]:
# Skapa en variabel för varje fysisk variabel

Undersök storleken på varje variabel, exempelvis med `print(f.shape)`.
Försök förstå vad de olika dimensionerna betyder.
Exempelvis: vilken dimension i `vmr` motsvarar höjd?

In [ ]:
# Undersök variablerna

### Steg 2

Importera modulen `olr` i mappen `Kod` och namnge den `olr`:

In [ ]:
import Kod.olr as olr

Med till exempel `help(olr)` kan du ta reda på vilka funktioner som finns i modulen och hur de fungerar.

In [ ]:
help(olr)

### Steg 3

Bli bekant med funktionen `olr.spectral_radiance`, exempelvis med `help(olr.spectral_radiance)`.
Hur anropar man funktionen, och vad returnerar den?

En parameter till funktionen är `za` för zenitvinkeln.
I vilken enhet förväntar sig funktionen att zenitvinkeln anges?

In [ ]:
# Bekanta dig med olr.spectral_radiance

Importera Matplotlib och plotta resultatet från `olr.spectral_radiance` för några utvalda zenitvinklar.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

<div class="alert alert-block alert-info">
    <b>Tips:</b>
    Får du <tt>RuntimeWarning</tt>-varningar och en array med <tt>nan</tt> och/eller <tt>inf</tt>?
    Kontrollera att alla parametrar till funktionen är passande och med korrekt enhet.
</div>

### Steg 4

Nu vet vi hur vi kan simulera den spektrala radiansen $I$ för en given zenitvinkel.
Nästa steg är att beräkna den spektrala emittansen $E_s$ enligt Ekvation (2):

$$
E_s(\nu) = C \cdot \int_{\theta=0}^{\pi/2} I(\nu, \theta)\cos(\theta)\sin(\theta) \, d\theta
\tag{2}
$$

där $C$ är konstanten du beräknade genom att integrera Ekvation (1) analytiskt,
och $I$ ges av funktionen `olr.spectral_radiance`.

För att beräkna integralen behöver du beräkna [integranden](https://sv.wiktionary.org/wiki/integrand) (uttrycket inuti integralen) för ett antal zenitvinklar $\theta$ inom integralens gränser.
Detta görs enklast med en loop.

**Notera:** $I$ från `olr.spectral_radiance` är en array med värden för alla frekvenser.
Därför blir även integranden en array.

Skapa en loop som beräknar integranden för ett antal zenitvinklar inom integralgränserna.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 5

Nu återstår det att utföra själva integrationen i Ekvation (2).
Integraler kan beräknas numeriskt med [trapetsregeln](https://sv.wikipedia.org/wiki/Trapetsregeln) via funktionen `np.trapezoid` (läs dokumentationen).

<div class="alert alert-block alert-success">
    <b>Info:</b>
    I äldre versioner av NumPy heter funktionen <tt>np.trapz</tt>.
    Om du får ett felmeddelande att <tt>np.trapezoid</tt> inte finns, prova <tt>np.trapz</tt> istället.
</div>

För att kunna använda `np.trapezoid` behöver du spara integrandens värde från varje iteration i loopen i Steg 4.
Integranden är en endimensionell array med längd $m$ (antalet frekvenser).
När du beräknar integranden för $n$ olika zenikvinklar (där du själv väljer $n$) får du $n$ sådana arrays.
Dessa kan sparas som rader i en tvådimensionell matris av storlek $n \times m$.

Börja med att skapa en tom $n \times m$ matris,
där $n$ är antalet zenitvinklar och $m$ är antalet frekvenser i $I$.

<div class="alert alert-block alert-info">
    <b>Tips:</b>
    Kommer du ihåg hur vi skapade en tom tvådimensionell array i Python-introduktionen?
    <details>
        <summary><em>Ledtråd:</em></summary>
        Titta på Problem 4 under Programmeringsproblem i Python-introduktionen.
    </details>
</div>

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Modifiera din loop från Steg 4 så att resultatet från iteration $i$ sparas i rad $i$ i matrisen du skapade.

<div class="alert alert-block alert-info">
    <b>Tips:</b>
    Börja med ett litet antal zenitvinklar så att koden körs snabbare.
    När du är nöjd med att allt fungerar kan du öka antalet zenitvinklar för att få en mer noggrann beräkning.
</div>

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 6

Beräkna nu integralen med `np.trapezoid`.
Eftersom du skickar in en tvådimensionell matris behöver du ange vilken axel (dimension) som ska integreras över med parametern `axis`.

<div class="alert alert-block alert-info">
<b>Tips: </b>
    Tänk på om du vill integrera över rader eller kolumner (zenitvinklar eller frekvenser).
    Fundera även på vad som ska vara <tt>y</tt> respektive <tt>x</tt> i anropet till <tt>np.trapezoid</tt>.
    <details>
        <summary><em>Ledtråd:</em></summary>
        Se Problem 5 under Programmeringsproblem i Python-introduktionen.
    </details>
</div>

Genom att multiplicera med konstanten $C$ får vi slutligen $E_s$.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Det är svårt i det här skedet att kontrollera om resultatet stämmer.
Detta är en vanlig utmaning inom programmering och dataanalys som man behöver lära sig att hantera.

För att göra det lättare för er får ni dock en liten ledtråd:
om du plottar den spektrala emittansen så borde de större värdena ha en storleksordning på 10<sup>-11</sup> Wm<sup>-2</sup>Hz<sup>-1</sup>.

### Steg 7

Vi kommer upprepa beräkningen av spektral emittans för olika förhållanden.
Implementera därför en funktion som utför uträkningen av $E_s$ (från Steg 5 och 6) och kalla den `spectral_exitance`.

Funktionen ska kunna användas för godtyckliga förhållanden: olika temperaturprofiler, vertikal sammansättning av gaser, osv.
Fundera på vilka parametrar funktionen behöver ta emot för att detta ska vara möjligt.

<div class="alert alert-block alert-info">
    <b>Tips:</b>
    Titta på vilka argument <tt>olr.spectral_radiance</tt> tar, och anpassa din funktion därefter.
</div>

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Testa funktionen med värdena från datafilen.
Kontrollera att resultatet stämmer överens med det du fick i föregående steg.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 8

Beräkna nu den totala emittansen genom att integrera över alla frekvenser enligt Ekvation (3):

$$
E = \int_{v=0}^\infty E_s(\nu) \, d\nu
\tag{3}
$$

Notera att det teoretiska intervallet går från 0 till oändligheten,
men här integrerar vi över alla frekvenser som ges i inputdatan.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Grattis! Du har nu beräknat OLR för jorden under de förhållanden som angavs i Avgränsningar. 🎉

Nu har du ett resultat som går att relatera till verkligheten.
Reflektera över om det värde du fick är rimligt.

### Steg 9

Implementera även en funktion för beräkningen i Steg 8 som du kallar `exitance`.
Denna funktion ska bland annat anropa `spectral_exitance` som du skapade i Steg 7 för att beräkna den spektrala emittansen.
Likt `spectral_exitance` ska den nya `exitance`-funktionen fungera för godtycklig temperaturprofil, vertikal sammansättning av gaser, osv.
Vilka parametrar behöver den här funktionen?

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Testa funktionen med värdena från datafilen.
Kontrollera att resultatet stämmer överens med det du fick i föregående steg.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 10

För att besvara frågorna behöver vi kunna störa temperatur- och vmr-profilen och se hur detta påverkar OLR.

Vi börjar med temperaturprofilen. Plotta först temperaturprofilen för standardvärdena från datafilen.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Nu vill vi störa temperaturprofilen genom att öka eller minska temperaturen i troposfären med ett angivet värde.
För enkelhets skull antar vi att temperaturdatan är ordnad från lägre till högre höjder och att tropopausen ligger där temperaturprofilen har sitt minsta värde.

Kopiera temperaturprofilen från datafilen och modifiera den så att temperaturen i troposfären (från marken upp till och med tropopausen) höjs med 10 K, medan stratosfären lämnas oförändrad från standardprofilen.

<div class="alert alert-block alert-info">
    <b>Tips:</b>
    Du kan använda <tt>copy</tt>-metoden för att skapa kopior på arrays, exempelvis: <tt>t_perturbed = t.copy()</tt><br>
    Fortfarande fast? Se ledtrådar nedan.
    <details>
        <summary><em>Ledtrådar:</em></summary>
        1. Om du behöver färska upp minnet om hur indexering och slices av NumPy arrays fungerar, gå tillbaka till Python-introduktionen.<br>
        2. Undersök i dokumentationen eller med din favoritsökmotor om NumPy har en funktion för att hitta indexet för det minsta värdet i en array.<br>
        3. Fortfarande fast? Det kan hjälpa att börja med Problem 6 under Programmeringsproblem i Python-introduktionen.
    </details>
</div>

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Definera därefter en funktion `perturb_t` för att störa temperaturprofilen:

```python
perturb_t(t, dt)
```
där `t` är originalprofilen och `dt` är störningen i K som läggs till troposfären.

<div class="alert alert-block alert-warning">
    <b>Varning:</b>
    Skapa en kopia av temperaturarrayen inuti funktionen innan du modifierar den, annars riskerar du att oavsiktligt ändra originaldatan.<br>
    (Detta beror på att Python skickar argument till funktioner som referenser snarare än kopior.)
</div>

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Kontrollera nu att din funktion stämmer genom att plotta temperaturprofilen för standardvärdena från datafilen och en stördditt temperaturprofilen i samma figur.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 11

Skapa en liknande funktion för volymandelar:

```python
perturb_vmr(igas, vmr, dvmr, t)
```
där `igas` anger indexet för gasen som ska störas,
`vmr` är matrisen med volymandelar,
och `dvmr` är en relativ störning (t.ex. 0.1 för 10% ökning).

Precis som `perturb_t` ska denna funktion endast störa värdena i troposfären.
Därför behöver funktionen även ta in `t` för att identifiera tropopausen.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Plotta några vmr-profiler, både ostörda och störda, för att kontrollera att funktionen fungerar som tänkt.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

## Frågor
Svaren till följande frågor ska lämnas in individuellt på Canvas.

<div class="alert alert-block alert-danger">
    <b>Observera:</b>
    Här måste du använda minst 100 zenitvinklar när du beräknar integralen i Ekvation (2).
    Annars finns det risk att resultaten påverkas av alltför stora numeriska fel.
</div>


<div class="alert  alert-block alert-info">
    <b>Tips:</b>
    Kommer du ihåg hur man sparar figurer? Om inte, se avsnittet om Matplotlib i Python-introduktionen.
</div>

### Del 1: Kodutveckling och testning

#### Fråga 1

Skapa en figur som visar spektral radians som funktion av vågtal för två olika zenitvinklar: 10° och 80°.

Märk kurvorna tydligt, t.ex. med en legend. För denna och alla kommande figurer: ange storhet och enhet på både x- och y-axeln.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 2

Plotta spektral emittans som funktion av vågtal. Glöm inte att ange enheter.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 3

Beräkna OLR (emittans) för standardvärdena i datafilen. Vilket värde får du?

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 4

Vad har OLR i föregående fråga för enhet? Ange svaret utan mellanslag (för att den automatiska rättningen ska fungera).

**Svar:** (Skriv ditt svar här om du vill)

#### Fråga 5

Hur kom du fram till att ditt beräknade OLR-värde är rimligt?

**Svar:** (Skriv ditt svar här om du vill)

#### Fråga 6

Skapa en figur som visar hur temperaturen varierar med höjden för standardvärden från datafilen.

Plotta även temperaturprofilen efter att du har ökat temperaturen med 1 K i troposfären.
Se till att de två kurvorna är tydligt märkta, exempelvis med en legend.

Observera att atmosfäriska profiler vanligtvis visualiseras med höjd på y-axeln och temperatur på x-axeln.
Använd samma upplägg här, så blir figuren enklare att jämföra med de som visats på föreläsningarna.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Del 2: Förändring i OLR

#### Fråga 7

CO<sub>2</sub>-halten har ökat med ca 47 % mellan preindustriella nivåer och 2019.

Plotta skillnaden i spektral emittans (som funktion av vågtal) när CO<sub>2</sub>-halten i troposfären ökar med 47 %.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 8

Vad är förändringen i OLR efter CO<sub>2</sub>-ökningen? Svara i samma enhet som i fråga 4 och var noga med tecknet.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 9

CH<sub>4</sub>-halten har ökat från ca 729 ppb år 1750 till 1866 ppb år 2019, vilket motsvarar en relativ ökning på 156%.

Plotta skillnaden i spektral emittans (som funktion av vågtal) när CH<sub>4</sub>-halten i troposfären ökar med 156%.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 10

Vad är förändringen i OLR efter CH<sub>4</sub>-ökningen? Svara i samma enhet som i fråga 4 och var noga med tecknet.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 11

Vad är den relativa ökningen i N<sub>2</sub>O-halten mellan 1750 och 2019?
Använd värden från [IPCC, 2021: Annex III](https://www.ipcc.ch/report/ar6/wg1/downloads/report/IPCC_AR6_WGI_AnnexIII.pdf) och ange svaret i procent (%).

**Svar:** (Skriv ditt svar här om du vill)

#### Fråga 12

Plotta skillnaden i spektral emittans (som funktion av vågtal) för N<sub>2</sub>O-förändringen mellan 1750 och 2019 i troposfären.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 13

Vad är förändringen i OLR efter förändringen i N<sub>2</sub>O-halten? Svara i samma enhet som i fråga 4 och var noga med tecknet.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 14

Förändringen i O<sub>3</sub>-halt varierar kraftigt mellan regioner, men den globala ökningen av troposfäriskt O<sub>3</sub> är i storleksordningen 30% sedan preindustriell tid.

Plotta skillnaden i spektral emittans (som funktion av vågtal) när O<sub>3</sub>-halten i troposfären ökar med 30%.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 15

Vad är förändringen i OLR efter O<sub>3</sub>-ökningen? Svara i samma enhet som i fråga 4 och var noga med tecknet.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 16

Skapa en graf som sammanfattar dina OLR-resultat för  CO<sub>2</sub>, CH<sub>4</sub>, N<sub>2</sub>O och O<sub>3</sub>, liknande [Figur 7.6](https://www.ipcc.ch/report/ar6/wg1/figures/chapter-7/figure-7-6) i IPCC, 2021: Kapitel 7. Climate Change 2021: The Physical Science Basis. Plotta även IPCC:s värden i samma graf för jämförelse.

Här har du en chans att visa din kreativitet! Tänk på följande:

- Det ska framgå tydligt vad som visas och vilka enheter som används.
- Figur 7.6 kombinerar CH4 och N2O i en stapel. Vi rekommenderar en stapel (eller motsvarande) per växthusgas.
- IPCC definierar strålningsdrivning (radiative forcing) så att positiva värden innebär uppvärmning, medan positiva förändringar i OLR innebär nedkylning. Du behöver därför byta tecken på antingen dina OLR-värden eller IPCC:s värden för att kunna jämföra dem.
- Figur 7.6 innehåller text till höger om grafen – denna behöver du inte inkludera.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 17

Sammanfatta kort dina resultat. Ligger de inom osäkerhetsintervallet för IPCC:s värden? Kommentera eventuella avvikelser.

**Svar:** (Skriv ditt svar här om du vill)

### Del 3: Klimatförändringar

#### Fråga 18

Öka en växthusgas i taget i troposfären med 1 %.
Vilken växthusgas ger störst förändring i OLR per procentenhet ökning, det vill säga är starkast vid relativa förändringar?

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 19

Hur mycket förändras OLR vid en fördubbling av CO<sub>2</sub>-halten i troposfären?
Svara i samma enhet som i fråga 4.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 20

Hur stor temperaturökning i troposfären krävs för att kompensera OLR-förändringen från en CO<sub>2</sub>-fördubbling och återställa strålningsbalansen?
Svara i K.

Tips: Du kan hitta temperaturvärdet med “trial and error”. Men kan du även beräkna den med någon fysikalisk lag?

In [ ]:
# Skriv din kod här, lägg till celler efter behov